## Classification Model that Predicts the Best Optimizer for Each Circuit

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
#All optimzation runs
master_df = pd.read_csv("./dbs/original_data/single_opt_master.csv")
master_df.head()

In [ ]:
# Adds a percentage two qubit gate reduction column to the master dataframe
master_df["two_qubit_gate_reduction_pct"] = (
    (master_df["initial_two_qubit_gates"] - master_df["final_two_qubit_gates"])
    / master_df["initial_two_qubit_gates"]
    * 100
)

In [ ]:
#Best optimization run for each circuit based on final two qubit depth
best_df = master_df.loc[
    master_df.groupby("original_circuit")["final_two_qubit_gates"].idxmin()
].copy()
best_df

In [ ]:
#Labels and features for the best optimization runs
y = best_df["opt_1"]
X = best_df[
    [
    "original_category",
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
    ]
]


In [ ]:
X = pd.get_dummies(
    X,
    columns=["original_category"],
    drop_first=False,
    dtype=int
)

In [ ]:
#Circuit names for mapping back to master df
circuit_names = best_df["original_circuit"]

In [ ]:
#Train test split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(X, y_encoded, circuit_names, test_size=0.2, random_state=42)


In [ ]:
print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])
print("Number of features:", X_train.shape[1])
print("Feature names:", X_train.columns.tolist())

# Random Forest Training and Evaluation

In [ ]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
rf_classifier = RandomForestClassifier(n_estimators=50, random_state=42)

In [ ]:
#Training rl classifier
rf_classifier.fit(X_train, y_train)

In [ ]:
#Training percentage
train_score = rf_classifier.score(X_train, y_train)
train_score

In [ ]:
#Testing percentage
test_score = rf_classifier.score(X_test, y_test)
test_score

In [ ]:
from sklearn.metrics import classification_report

preds = rf_classifier.predict(X_test)

print(classification_report(y_test, preds))

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    rf_classifier,
    X_test,
    y_test,
    display_labels=le.classes_,
    xticks_rotation=45,
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_classifier.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

In [ ]:
#Evaluating average 2Q gate count reduction on RF model
total_reduction = 0

optimizers = ['qiskit_ai', 'qiskit_standard', 'tket', 'wisq_bqskit', 'wisq_rules']

for i, (_, row) in enumerate(X_test.iterrows()):
    pred = rf_classifier.predict(row.to_frame().T)[0]
    prediction = le.inverse_transform([pred])[0]
    circuit_name = names_test.iloc[i]

    pred_row = master_df[
        (master_df["original_circuit"] == circuit_name)
        & (master_df["opt_1"] == prediction)
    ]

    original_gates = pred_row["initial_two_qubit_gates"].iloc[0]
    final_gates = pred_row["final_two_qubit_gates"].iloc[0]

    reduction_pct = (
        (original_gates - final_gates)
        / original_gates
        * 100
    )

    total_reduction += reduction_pct

average_reduction_RF = total_reduction / len(X_test)

print("Average reduction:", average_reduction_RF)

# XGBoost Training and Evaluation

In [ ]:
from xgboost import XGBClassifier
xgb_classifier = XGBClassifier(
    n_estimators=50, 
    learning_rate=0.05,
    objective="multi:softmax",
    random_state=42
)

In [ ]:
xgb_classifier.fit(X_train, y_train)

In [ ]:
#Training percentage
train_score = xgb_classifier.score(X_train, y_train)
train_score

In [ ]:
#Testing percentage
test_score = xgb_classifier.score(X_test, y_test)
test_score

In [ ]:
preds = xgb_classifier.predict(X_test)

print(classification_report(y_test, preds))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_estimator(
    xgb_classifier,
    X_test,
    y_test,
    display_labels=le.classes_,
    xticks_rotation=45,
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_classifier.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

In [ ]:
#Evaluating average 2Q gate count reduction on RF model
total_reduction = 0

optimizers = ['qiskit_ai', 'qiskit_standard', 'tket', 'wisq_bqskit', 'wisq_rules']

for i, (_, row) in enumerate(X_test.iterrows()):
    pred = xgb_classifier.predict(pd.DataFrame([row]))[0]
    prediction = le.inverse_transform([pred])[0]
    circuit_name = names_test.iloc[i]

    pred_row = master_df[
        (master_df["original_circuit"] == circuit_name)
        & (master_df["opt_1"] == prediction)
    ]

    original_gates = pred_row["initial_two_qubit_gates"].iloc[0]
    final_gates = pred_row["final_two_qubit_gates"].iloc[0]

    reduction_pct = (
        (original_gates - final_gates)
        / original_gates
        * 100
    )

    total_reduction += reduction_pct

average_reduction_RF = total_reduction / len(X_test)

print("Average reduction:", average_reduction_RF)

# Evaluating using basic methods

###            Single best optimizer (wisq bqskit)

In [ ]:
#Average gate reduction on test set of just using wisq_bqskit
test_circuits = set(names_test)

wisq_test_avg_reward = (
    master_df[
        (master_df["opt_1"] == "wisq_bqskit")
        & (master_df["original_circuit"].isin(test_circuits))
    ]["two_qubit_gate_reduction_pct"]
    .mean()
)

print(wisq_test_avg_reward)

### Random Optimizer

In [ ]:
#Average gate reduction using random optimizer
import random
optimizers = ["wisq_bqskit", "wisq_rules", "qiskit_standard", "qiskit_ai", "tket"]
total_reduction = 0
for i, (_, row) in enumerate(X_test.iterrows()):
    
    random_optimizer = random.choice(optimizers)
    circuit_name = names_test.iloc[i]

    pred_row = master_df[
        (master_df["original_circuit"] == circuit_name)
        & (master_df["opt_1"] == random_optimizer)
    ]

    original_gates = pred_row["initial_two_qubit_gates"].iloc[0]
    final_gates = pred_row["final_two_qubit_gates"].iloc[0]

    reduction_pct = (
        (original_gates - final_gates)
        / original_gates
        * 100
    )

    total_reduction += reduction_pct

average_reduction_rand_opt = total_reduction / len(X_test)

print("Average reduction:", average_reduction_rand_opt)